In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# 1. 경로 설정
base_model_path = "Qwen/Qwen3-4B"  # 베이스 모델명 또는 경로
lora_adapter_path = "/data/ephemeral/pro-nlp-finalproject-nlp-13/src/models/qwen_dpo/final_model/policy"      # LoRA 학습 결과가 저장된 폴더
save_to = "src/models/merged_model/qwen_dpo_merged_model"           # 병합된 모델이 저장될 경로

print("🚀 베이스 모델 로드 중...")
# V100 환경이시라면 torch_dtype=torch.float16을 권장합니다.
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.float16,
    device_map="cpu", # 병합 시에는 CPU 메모리를 쓰는 것이 안전합니다 (RAM 부족 시 확인 필요)
    trust_remote_code=True
)


model = PeftModel.from_pretrained(model, lora_adapter_path)

merged_model = model.merge_and_unload()

print(f"💾 병합된 모델 저장 중 -> {save_to}")
merged_model.save_pretrained(save_to, safe_serialization=True)

# 토크나이저도 함께 저장해야 나중에 바로 쓸 수 있습니다.
tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=True)
tokenizer.save_pretrained(save_to)

print("✅ 모든 작업이 완료되었습니다!")

🚀 베이스 모델 로드 중...


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

💾 병합된 모델 저장 중 -> src/models/merged_model/qwen_dpo_merged_model
✅ 모든 작업이 완료되었습니다!
